In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
from pathlib import Path
# Procura a raiz do projeto (pasta que contém `src`) subindo na arvore
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'src').is_dir():
            return p
    return start
proj_root = str(find_project_root())
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)
print('Project root added to sys.path:', proj_root)

from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import LabelBinarizer, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from src.preprocessing import SavgolFilter, SNV, make_savgol, make_scaler, make_snv
from src.utils import extrair_numero
from src.models import PLSDAClassifier, PLSDAMulticlass

Project root added to sys.path: c:\Users\Pedro\Downloads\LIBS_NEW


In [2]:
from src.config import *

list_dirs()


BASE_DIR:  C:\Users\Pedro\Downloads\LIBS_NEW 
DATA_DIR:  C:\Users\Pedro\Downloads\LIBS_NEW\dataset 
MODELS_DIR:  C:\Users\Pedro\Downloads\LIBS_NEW\models 
PLOTS_DIR: C:\Users\Pedro\Downloads\LIBS_NEW\plots


In [7]:
import os

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

from src.training import run_experiment
from src.config import *

print('\nLendo dataset...')
file_path = Path(proj_root) / 'dataset/harinas_new_libs_b.csv'

df = pd.read_csv(file_path, sep=';', header=None)

y_raw = df.iloc[0, 1:].values.astype(str)
X_full = df.iloc[1:, 1:].values.astype(float).T

# O ultimo bloco numerico identifica o disco, mas nao faz parte da classe.
# A label completa continua sendo usada como grupo no SGKF.
classes_argentinas = {'A', 'B', 'C', 'D', 'E'}
classes_brasileiras = {'F', 'G', 'H', 'I', 'J', 'K', 'M', 'N'}

X = []
y = []
groups = []
ignored_labels = []

for xi, yi in zip(X_full, y_raw):
    sample_name, separator, disk = yi.rpartition('_')
    if not separator or not disk.isdigit() or not sample_name:
        raise ValueError(f"Label invalida: '{yi}'.")

    if sample_name in classes_argentinas:
        class_name = 'Argentina'
    elif sample_name in classes_brasileiras or (
        sample_name.startswith('E')
        and len(sample_name) > 1
        and sample_name[1].isdigit()
    ):
        class_name = 'Brasileira'
    else:
        # Amostras fora das duas classes, como BRANCO, nao entram no experimento.
        ignored_labels.append(yi)
        continue

    X.append(xi)
    y.append(class_name)
    groups.append(yi)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

print(f'Formato do dataset: {X.shape}')
print(f'Classes: {np.unique(y)}')
print(f'Grupos/discos: {np.unique(groups)}')
if ignored_labels:
    print(f'Labels ignoradas: {np.unique(ignored_labels)}')

unique_groups = np.unique(groups)
if len(unique_groups) < 2:
    raise ValueError('É necessário pelo menos 2 grupos para usar StratifiedGroupKFold.')

# Cada fold deve receber um grupo/disco distinto de cada classe quando possível.
groups_per_class = (
    pd.DataFrame({'class': y, 'group': groups})
    .drop_duplicates()
    .groupby('class')['group']
    .nunique()
)
n_splits = min(3, len(unique_groups), int(groups_per_class.min()))
if n_splits < 2:
    raise ValueError('É necessário pelo menos 2 discos por classe para usar StratifiedGroupKFold.')

sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

preprocessamentos = {
    'snv': [make_snv()],
    'standard_scaler': [make_scaler()],
}

modelos = {
    'svm_linear': SVC(kernel='linear', gamma='scale'),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42),
}

output_models = f'{BASE_DIR}/temp/models_groupkfold/farinha'
output_plots = f'{BASE_DIR}/temp/plots_groupkfold/farinha'

os.makedirs(output_models, exist_ok=True)
os.makedirs(output_plots, exist_ok=True)

class_labels = np.array(['Argentina', 'Brasileira'])
print(f'Executando Cross Validation com StratifiedGroupKFold ({n_splits} folds)...')
results_data = run_experiment(
    X=X,
    y=y,
    preprocessamentos=preprocessamentos,
    modelos=modelos,
    cv=sgkf,
    output_models_dir=output_models,
    output_plots_dir=output_plots,
    laser=1,
    labels=class_labels,
    experiment_name='stratified_groupkfold_farinha',
    display_labels=class_labels,
    groups=groups,
    summary_filename='all_pipelines_summary_farinha.json',
    ranking_filename='ranking_final_groupkfold_farinha.png',
)

results = results_data['results']
print('==============================')
print('RANKING FINAL')
print('==============================')

for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f'{k}: {v:.4f}')

print(f"Resumo consolidado salvo em: {results_data['summary_path']}")
print(f"Plot final salvo em: {results_data['ranking_path']}")
print('Processo concluído!')


Lendo dataset...


C:\Users\Pedro\AppData\Local\Temp\ipykernel_15456\2483062290.py:14: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254

Formato do dataset: (756, 12288)
Classes: ['Argentina' 'Brasileira']
Grupos/discos: ['A_1' 'A_2' 'A_3' 'B_1' 'B_2' 'B_3' 'C_1' 'C_2' 'C_3' 'D_1' 'D_2' 'D_3'
 'E11_1' 'E11_2' 'E11_3' 'E1_13_1' 'E1_13_2' 'E1_13_3' 'E1_19_1' 'E1_19_2'
 'E1_19_3' 'E26_1' 'E26_2' 'E26_3' 'E32_1' 'E32_2' 'E32_3' 'E69_1' 'E69_2'
 'E69_3' 'E84_1' 'E84_2' 'E84_3' 'E90_1' 'E90_2' 'E90_3' 'E98_1' 'E98_2'
 'E98_3' 'E_1' 'E_2' 'E_3' 'F_1' 'F_2' 'F_3' 'G_1' 'G_2' 'G_3' 'H_1' 'H_2'
 'H_3' 'I_1' 'I_2' 'I_3' 'J_1' 'J_2' 'J_3' 'K_1' 'K_2' 'K_3' 'M_1' 'M_2'
 'M_3']
Labels ignoradas: ['BRANCO_1' 'BRANCO_2' 'BRANCO_3' 'L_1' 'L_2' 'L_3']
Executando Cross Validation com StratifiedGroupKFold (3 folds)...
RANKING FINAL
snv_svm_linear: 0.9603
standard_scaler_svm_linear: 0.9392
snv_random_forest: 0.8929
standard_scaler_random_forest: 0.8929
Resumo consolidado salvo em: C:\Users\Pedro\Downloads\LIBS_NEW\temp\plots_groupkfold\farinha\all_pipelines_summary_farinha.json
Plot final salvo em: C:\Users\Pedro\Downloads\LIBS_NEW\temp\plo

In [8]:
# Experimento alternativo: o SGKF agrupa pela amostra-base, nao pelo disco.
# Assim, todos os discos de E1_13 ficam no mesmo grupo.

X_classes = []
y_classes = []
groups_classes = []
ignored_labels_classes = []

for xi, yi in zip(X_full, y_raw):
    sample_name, separator, disk = yi.rpartition('_')
    if not separator or not disk.isdigit() or not sample_name:
        raise ValueError(f"Label invalida: '{yi}'.")

    if sample_name in classes_argentinas:
        class_name = 'Argentina'
    elif sample_name in classes_brasileiras or (
        sample_name.startswith('E')
        and len(sample_name) > 1
        and sample_name[1].isdigit()
    ):
        class_name = 'Brasileira'
    else:
        ignored_labels_classes.append(yi)
        continue

    X_classes.append(xi)
    y_classes.append(class_name)
    groups_classes.append(sample_name)

X_classes = np.array(X_classes)
y_classes = np.array(y_classes)
groups_classes = np.array(groups_classes)

print(f'Formato do dataset: {X_classes.shape}')
print(f'Classes de classificacao: {np.unique(y_classes)}')
print(f'Amostras usadas como grupos: {np.unique(groups_classes)}')
if ignored_labels_classes:
    print(f'Labels ignoradas: {np.unique(ignored_labels_classes)}')

unique_class_groups = np.unique(groups_classes)
if len(unique_class_groups) < 2:
    raise ValueError('É necessário pelo menos 2 amostras para usar StratifiedGroupKFold.')

groups_per_class = (
    pd.DataFrame({'class': y_classes, 'group': groups_classes})
    .drop_duplicates()
    .groupby('class')['group']
    .nunique()
)
n_splits_classes = min(
    3,
    len(unique_class_groups),
    int(groups_per_class.min()),
)
if n_splits_classes < 2:
    raise ValueError('É necessário pelo menos 2 amostras por classe para usar StratifiedGroupKFold.')

sgkf_classes = StratifiedGroupKFold(
    n_splits=n_splits_classes,
    shuffle=True,
    random_state=42,
)

print(
    f'Executando SGKF por amostra-base ({n_splits_classes} folds), '
    'mantendo todos os discos da mesma amostra no mesmo fold...'
)
results_data_classes = run_experiment(
    X=X_classes,
    y=y_classes,
    preprocessamentos=preprocessamentos,
    modelos=modelos,
    cv=sgkf_classes,
    output_models_dir=f'{BASE_DIR}/temp/models_groupkfold/farinha_classes',
    output_plots_dir=f'{BASE_DIR}/temp/plots_groupkfold/farinha_classes',
    laser=1,
    labels=np.array(['Argentina', 'Brasileira']),
    experiment_name='stratified_groupkfold_por_amostra_base',
    display_labels=np.array(['Argentina', 'Brasileira']),
    groups=groups_classes,
    summary_filename='all_pipelines_summary_por_amostra_base.json',
    ranking_filename='ranking_final_groupkfold_por_amostra_base.png',
)

print('==============================')
print('RANKING FINAL - SGKF POR AMOSTRA-BASE')
print('==============================')
for pipeline_name, accuracy in sorted(
    results_data_classes['results'].items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(f'{pipeline_name}: {accuracy:.4f}')

print(f"Resumo consolidado salvo em: {results_data_classes['summary_path']}")
print(f"Plot final salvo em: {results_data_classes['ranking_path']}")

Formato do dataset: (756, 12288)
Classes de classificacao: ['Argentina' 'Brasileira']
Amostras usadas como grupos: ['A' 'B' 'C' 'D' 'E' 'E11' 'E1_13' 'E1_19' 'E26' 'E32' 'E69' 'E84' 'E90'
 'E98' 'F' 'G' 'H' 'I' 'J' 'K' 'M']
Labels ignoradas: ['BRANCO_1' 'BRANCO_2' 'BRANCO_3' 'L_1' 'L_2' 'L_3']
Executando SGKF por amostra-base (3 folds), mantendo todos os discos da mesma amostra no mesmo fold...
RANKING FINAL - SGKF POR AMOSTRA-BASE
standard_scaler_svm_linear: 0.9259
snv_svm_linear: 0.8995
standard_scaler_random_forest: 0.8717
snv_random_forest: 0.8228
Resumo consolidado salvo em: C:\Users\Pedro\Downloads\LIBS_NEW\temp\plots_groupkfold\farinha_classes\all_pipelines_summary_por_amostra_base.json
Plot final salvo em: C:\Users\Pedro\Downloads\LIBS_NEW\temp\plots_groupkfold\farinha_classes\ranking_final_groupkfold_por_amostra_base.png


In [11]:
# Exemplo de como as amostras-base sao distribuidas em cada fold.
# Os discos da mesma amostra aparecem juntos no mesmo grupo.

print('DISTRIBUICAO DAS AMOSTRAS POR FOLD')
print('===================================')

for fold_number, (train_idx, validation_idx) in enumerate(
    sgkf_classes.split(X_classes, y_classes, groups_classes),
    start=1,
):
    train_groups = np.unique(groups_classes[train_idx])
    validation_groups = np.unique(groups_classes[validation_idx])

    overlap = np.intersect1d(train_groups, validation_groups)
    if len(overlap) > 0:
        raise AssertionError(
            f'Amostras presentes em treino e validacao no fold {fold_number}: {overlap}'
        )

    def format_groups(group_names):
        formatted = []
        for group_name in group_names:
            group_class = np.unique(
                y_classes[groups_classes == group_name]
            )
            formatted.append(f'{group_name} ({group_class[0]})')
        return formatted

    print(f'Etapa {fold_number}')
    print(f'Treino ({len(train_groups)} amostras-base):')
    print(', '.join(format_groups(train_groups)))
    print(f'Validação ({len(validation_groups)} amostras-base):')
    print(', '.join(format_groups(validation_groups)))

DISTRIBUICAO DAS AMOSTRAS POR FOLD
Etapa 1
Treino (14 amostras-base):
B (Argentina), C (Argentina), D (Argentina), E11 (Brasileira), E1_19 (Brasileira), E26 (Brasileira), E69 (Brasileira), E84 (Brasileira), E90 (Brasileira), F (Brasileira), G (Brasileira), H (Brasileira), J (Brasileira), K (Brasileira)
Validação (7 amostras-base):
A (Argentina), E (Argentina), E1_13 (Brasileira), E32 (Brasileira), E98 (Brasileira), I (Brasileira), M (Brasileira)
Etapa 2
Treino (14 amostras-base):
A (Argentina), C (Argentina), E (Argentina), E11 (Brasileira), E1_13 (Brasileira), E1_19 (Brasileira), E32 (Brasileira), E84 (Brasileira), E90 (Brasileira), E98 (Brasileira), H (Brasileira), I (Brasileira), K (Brasileira), M (Brasileira)
Validação (7 amostras-base):
B (Argentina), D (Argentina), E26 (Brasileira), E69 (Brasileira), F (Brasileira), G (Brasileira), J (Brasileira)
Etapa 3
Treino (14 amostras-base):
A (Argentina), B (Argentina), D (Argentina), E (Argentina), E1_13 (Brasileira), E26 (Brasileira), E3

In [12]:
# Exemplo de como os discos sao distribuidos em cada fold.
# Cada grupo representa uma label completa, como E1_13_1.

print('DISTRIBUICAO DOS DISCOS POR FOLD')
print('=================================')

for fold_number, (train_idx, validation_idx) in enumerate(
    sgkf.split(X, y, groups),
    start=1,
):
    train_groups = np.unique(groups[train_idx])
    validation_groups = np.unique(groups[validation_idx])

    overlap = np.intersect1d(train_groups, validation_groups)
    if len(overlap) > 0:
        raise AssertionError(
            f'Discos presentes em treino e validacao no fold {fold_number}: {overlap}'
        )

    def format_disk_groups(group_names):
        formatted = []
        for group_name in group_names:
            group_class = np.unique(y[groups == group_name])
            formatted.append(f'{group_name} ({group_class[0]})')
        return formatted

    print(f'\nFOLD {fold_number}')
    print(f'Treino ({len(train_groups)} discos):')
    print(', '.join(format_disk_groups(train_groups)))
    print(f'Validação ({len(validation_groups)} discos):')
    print(', '.join(format_disk_groups(validation_groups)))

DISTRIBUICAO DOS DISCOS POR FOLD

FOLD 1
Treino (42 discos):
A_2 (Argentina), A_3 (Argentina), B_1 (Argentina), B_3 (Argentina), C_2 (Argentina), C_3 (Argentina), D_2 (Argentina), D_3 (Argentina), E11_2 (Brasileira), E11_3 (Brasileira), E1_13_2 (Brasileira), E1_13_3 (Brasileira), E1_19_2 (Brasileira), E1_19_3 (Brasileira), E26_1 (Brasileira), E26_3 (Brasileira), E32_2 (Brasileira), E32_3 (Brasileira), E69_2 (Brasileira), E69_3 (Brasileira), E84_2 (Brasileira), E84_3 (Brasileira), E90_2 (Brasileira), E90_3 (Brasileira), E98_2 (Brasileira), E98_3 (Brasileira), E_1 (Argentina), E_3 (Argentina), F_1 (Brasileira), F_3 (Brasileira), G_1 (Brasileira), G_3 (Brasileira), H_2 (Brasileira), H_3 (Brasileira), I_2 (Brasileira), I_3 (Brasileira), J_2 (Brasileira), J_3 (Brasileira), K_1 (Brasileira), K_3 (Brasileira), M_2 (Brasileira), M_3 (Brasileira)
Validação (21 discos):
A_1 (Argentina), B_2 (Argentina), C_1 (Argentina), D_1 (Argentina), E11_1 (Brasileira), E1_13_1 (Brasileira), E1_19_1 (Brasilei